In [14]:
# Import necessary libraries

import os

from langchain_community.utilities import SQLDatabase

from langchain_classic.chains import create_sql_query_chain

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from langchain_google_genai import ChatGoogleGenerativeAI

In [15]:
# Connect your MySQL database
# Make sure to install the required packages
host = 'localhost'
port = '3306'
username = 'root'
password = '5998'
database_schema = 'text_to_sql'
mysql_uri = f"mysql+pymysql://{username}:{password}@{host}:{port}/{database_schema}"
db = SQLDatabase.from_uri(mysql_uri, sample_rows_in_table_info=2)

In [16]:
# Database connection
db = SQLDatabase.from_uri(mysql_uri, sample_rows_in_table_info=1)

In [17]:
# create a llm propt template
# Create the LLM Prompt Template
from langchain_core.prompts import ChatPromptTemplate

template = """Based on the table schema below, write a SQL query that would answer the user's question:
Remember : Only provide me the sql query dont include anything else. Provide me sql query in a single line dont add line breaks
Table Schema: {schema}
Question: {question}
SQL Query:
"""

prompt = ChatPromptTemplate.from_template(template)

In [18]:
# get the schema of the database
def get_schema(db):
    schema = db.get_table_info()
    return schema

In [19]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")

In [20]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    google_api_key=api_key
)

In [21]:
# Create the SQL query chain using the LLM and the prompt template

sql_chain = (
    RunnablePassthrough.assign(schema=lambda _:get_schema(db)) 
    | prompt
    | llm.bind(stop=["\nSQLResult:"])
    | StrOutputParser()
)

In [22]:
# test the SQL query chain with a sample question

resp = sql_chain.invoke({"question": "What is the total 'Line Total' for Geiss Company"})
print(resp)

SELECT SUM(T1.`Line Total`) FROM sales_order AS T1 JOIN customers AS T2 ON T1.`Customer Name Index` = T2.`Customer Index` WHERE T2.`Customer Names` = 'Geiss Company'


In [23]:
#test the SQL query chain with a sample question
resp=sql_chain.invoke({"question": "What was the budget of Product 12"})
print(resp)

SELECT `2017 Budgets` FROM `2017_budgets` WHERE `Product Name` = 'Product 12'


In [24]:
import re

query = re.search(r"```sql\s*(.*?)\s*```", resp, re.DOTALL | re.IGNORECASE)

if query:
    query=query.group(1).strip()

In [25]:
import re

question = "Show all records from the sales_order table"

schema = db.get_table_info()

response = llm.invoke(
    prompt.format(
        schema=schema,
        question=question
    )
)

resp = response.content

# Extract text from Gemini response
if isinstance(resp, list):
    if len(resp) > 0 and isinstance(resp[0], dict):
        resp = resp[0].get("text", "")
    else:
        resp = str(resp)

# Convert to string
resp = str(resp)

# Extract SQL from markdown code block if present
match = re.search(
    r"```(?:sql)?\s*(.*?)\s*```",
    resp,
    re.DOTALL | re.IGNORECASE
)

if match:
    query = match.group(1).strip()
else:
    query = resp.strip()

print("Generated SQL Query:")
print(query)

Generated SQL Query:
SELECT * FROM sales_order


### RAGAS Implementation


In [27]:
import sys
import importlib.metadata as md

print("Python executable:")
print(sys.executable)
print()

packages = [
    "ragas",
    "langchain",
    "langchain-core",
    "langchain-community",
    "langchain-google-genai"
]

for package in packages:
    try:
        print(f"{package}: {md.version(package)}")
    except md.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

Python executable:
C:\Users\rajem\anaconda3\python.exe

ragas: 0.4.3
langchain: 1.4.0
langchain-core: 1.6.3
langchain-community: 0.4.2
langchain-google-genai: 4.4.0


In [28]:
import sys

!{sys.executable} -m pip check

mlxtend 0.24.0 has requirement scipy>=1.16.3, but you have scipy 1.15.0.
numba 0.61.0 has requirement numpy<2.2,>=1.24, but you have numpy 2.4.3.
sklearn-compat 0.1.3 has requirement scikit-learn<1.7,>=1.2, but you have scikit-learn 1.8.0.
streamlit 1.45.1 has requirement pandas<3,>=1.4.0, but you have pandas 3.0.1.
streamlit 1.45.1 has requirement pillow<12,>=7.1.0, but you have pillow 12.3.0.


In [26]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

ModuleNotFoundError: No module named 'langchain_community.chat_models.vertexai'

In [ ]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

In [ ]:
evaluator_llm = LangchainLLMWrapper(llm)
evaluator_embeddings = LangchainEmbeddingsWrapper(embeddings)